# CS781 Phase 6 - Abstractive legal summarization (one notebook, four runs on the four Phase-1 models)

| Run | Phase-1 model | Technique | Role |
|---|---|---|---|
| 1 | Arch 4 | **LoRMA** (compulsory) | also the KD teacher |
| 2 | Arch 3 | Houlsby bottleneck adapters | |
| 3 | Arch 2 | LoRA | |
| 4 | Arch 1 | **Knowledge distillation** (half-depth student, CE + KL to the Run-1 teacher) | mandatory KD experiment |
| 5 | Arch 1 | same half-depth student, no teacher (optional) | isolates what KD adds |

**Method.** The Phase-1 models are decoder-only LMs, so summarization is conditional generation:
`[selected document tokens] [SEP] [summary] [END]`, loss on the summary tokens only, greedy decoding with no-repeat 3-grams.
Long documents: sentences are scored, and the token budget is filled with lead + tail + SumBasic-ranked middle sentences in original order
(strategies `select | spread | headtail | lead`). Only Phase-1 weights are used; no external model or API.

**Kaggle inputs to attach (Add Input)**
1. The Phase 6 data (folder `Phase_6_Summ` with `Train/train.json` and `Test/test.json`).
2. The four Phase-1 checkpoint datasets, exactly as in Phase 4/5 (Arch 1, 2, 3, 4, each with its checkpoint and `custom_bpe.json`).
3. `phase4-allrun-v4.ipynb` (as a dataset, or the Phase-4 notebook via Add Input): cells 1-7 are reused for the model code.
4. Optional: the committed Phase-4/5 output containing `diagnostics.json` (skips the pre-tokeniser/variant search).
5. Run 4 needs the Run-1 teacher file `Run1_Arch4_LoRMA_trainable.pt`: same session as Run 1, or attach the committed output of the session that ran it.
6. GPU on (T4 x2 or P100). Internet is optional: only BERTScore on Dev needs it (`pip install bert-score` + model download); otherwise Dev reports ROUGE only.

**How to run.** Keep `P6_DRY_RUN = True` first (about 15-20 min, tiny data, 3 min per run). Then set it to `False`.
To use two GPU sessions at once: session A `P6_RUNS = [1, 4]`, session B `P6_RUNS = [2, 3]`; outputs go to `/kaggle/working/p6_out` (Save & Commit), zips are standalone.
Submissions: `p6_out/submission_<run>.zip` (each holds `predictions.json`, list format).

In [ ]:
# 1. Setup and switches
import os, re, ast, json, glob, math, time, random, zipfile, pickle, gc, collections, sys
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

P6_DRY_RUN = True                 # True: tiny end-to-end smoke test (~15-20 min). Set False for the real run.
P6_RUNS = [1, 2, 3, 4]            # which runs this session executes (5 = optional no-KD baseline for the KD comparison)
P6_MINUTES = {1: 45, 2: 40, 3: 40, 4: 40, 5: 30}   # wall-clock training budget per run (the LR schedule follows it)
P6_STRATEGY = "select"            # long-document strategy: select | spread | headtail | lead
P6_STRATEGY_ABLATION = True       # after Run 1, score the other strategies on Dev (inference-time comparison)
P6_SKIP_DONE = True               # skip a run whose submission already exists in the output folder
P6_DEV_N = 150                    # held-out validation documents cut from train.json
P6_NO_REPEAT = 3                  # no-repeat n-gram size at decoding (0 = off)
P6_MICRO_BS, P6_ACCUM = 8, 2      # micro-batch and gradient accumulation (OOM auto-halves the micro-batch)
P6_GEN_BS = 32
P6_BERT_MODEL = "roberta-large"   # BERTScore needs internet (pip + model download); skipped automatically if unavailable
P6_SEED = 42
P6_OUT = "/kaggle/working/p6_out"
os.makedirs(P6_OUT, exist_ok=True)

random.seed(P6_SEED); np.random.seed(P6_SEED); torch.manual_seed(P6_SEED)
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU - attach a GPU accelerator")
print("DRY_RUN" if P6_DRY_RUN else "FULL RUN", "| runs this session:", P6_RUNS)

In [ ]:
# 2. Re-use the Phase-1 model code from the Phase-4 notebook (config, checkpoint discovery, tokenizer,
#    architectures 1-4, lenient loader) so the four Phase-1 models are rebuilt exactly as in Phases 4/5.
#    Attach phase4-allrun-v4.ipynb (as a dataset or via Add Input) and the four Phase-1 checkpoint datasets.
PHASE4_NOTEBOOK = None            # path to the Phase-4 .ipynb; None = search /kaggle/input and /kaggle/working
P4_BOOT_CELLS = (1, 2, 3, 4, 5, 6)
P4_REQUIRED = ("build_backbone", "pretokenize", "tokenizer", "lm_ppl", "ARCH_OPTS", "PREPROC_MODES")

def _p4_candidates():
    hits = glob.glob("/kaggle/input/**/*.ipynb", recursive=True) + glob.glob("/kaggle/working/**/*.ipynb", recursive=True)
    hits = [h for h in hits if not re.search(r"phase.?[56]|lsi|summ", os.path.basename(h), re.I)]
    hits.sort(key=lambda h: (not re.search(r"phase.?4|allrun|cjpe", h, re.I), not re.search(r"v4", h, re.I), len(h)))
    return hits

def _p4_cells(path):
    nb = json.load(open(path, encoding="utf-8")); out = {}
    for c in nb.get("cells", []):
        if c.get("cell_type") != "code": continue
        src = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        head = "\n".join(src.splitlines()[:5])
        m = re.search(r"^#\s*(?:CELL\s*)?(\d+)\.\s", head, re.M | re.I)
        if m and int(m.group(1)) not in out: out[int(m.group(1))] = src
    return out

def _sanitize(src):
    lines = []
    for l in src.splitlines():
        s = l.lstrip()
        lines.append(l[:len(l) - len(s)] + "pass  # [P6 skipped] " + s if s.startswith(("!", "%")) else l)
    return "\n".join(lines)

P4_ERRORS = []
if PHASE4_NOTEBOOK != "skip":
    _path = PHASE4_NOTEBOOK or next(iter(_p4_candidates()), None)
    assert _path, "Phase-4 notebook not found: attach phase4-allrun-v4.ipynb (Add Input) or set PHASE4_NOTEBOOK"
    print("Phase-4 notebook:", _path)
    _cells = _p4_cells(_path)
    print("numbered code cells found:", sorted(_cells))
    for _k in P4_BOOT_CELLS:
        if _k not in _cells: print(f"  cell {_k} missing"); continue
        try:
            exec(compile(_sanitize(_cells[_k]), f"<phase4 cell {_k}>", "exec"), globals())
        except Exception as e:
            P4_ERRORS.append((_k, repr(e))); print(f"  phase-4 cell {_k} raised: {repr(e)[:300]}")
    if 7 in _cells:   # cell 7 holds build_backbone / lm_ppl; take its definitions only, not the diagnosis run
        _tree = ast.parse(_sanitize(_cells[7])); _keep = []
        for _n in _tree.body:
            if isinstance(_n, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "DIAG" for t in _n.targets): break
            if isinstance(_n, (ast.FunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom, ast.Assign)): _keep.append(_n)
        try:
            exec(compile(ast.Module(body=_keep, type_ignores=[]), "<phase4 cell 7 defs>", "exec"), globals())
        except Exception as e:
            P4_ERRORS.append((7, repr(e))); print("  phase-4 cell 7 definitions raised:", repr(e)[:300])
_missing = [n for n in P4_REQUIRED if n not in globals()]
assert not _missing, f"missing after import: {_missing}. Errors: {P4_ERRORS}. Paste Phase-4 cells 1-7 above and set PHASE4_NOTEBOOK='skip'."
device = globals().get("device", device)
P6_AMP_DTYPE = globals().get("AMP_DTYPE", torch.float16)
P6_AMP_ON = bool(globals().get("USE_AMP", True)) and device.type == "cuda"
print("checkpoints:", {a: bool(globals().get(f"ARCH{a}_CHECKPOINT")) for a in (1, 2, 3, 4)})

# diagnostics.json from a committed Phase-4/5 run (optional): saves the pre-tokeniser / variant search below
P4_DIAG = None
for _p in glob.glob("/kaggle/input/**/diagnostics.json", recursive=True) + glob.glob("/kaggle/working/**/diagnostics.json", recursive=True):
    try:
        _d = json.load(open(_p))
        if "preproc" in _d: P4_DIAG = _d; print("diagnostics.json:", _p, "| preproc =", _d["preproc"]); break
    except Exception: pass

In [ ]:
# 3. Phase-6 data: train.json -> train / held-out Dev, test.json
def p6_find(name):
    c = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    pref = [p for p in c if re.search(r"phase.?6|summ", p, re.I)]
    return (pref or c or [None])[0]

def p6_read_records(path):
    txt = open(path, encoding="utf-8").read()
    try: raw = json.loads(txt)
    except json.JSONDecodeError: raw = [json.loads(l) for l in txt.splitlines() if l.strip()]
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): raw = raw["data"]
        else: raw = [{"id": k, **(v if isinstance(v, dict) else {"document": v})} for k, v in raw.items()]
    recs = []
    for i, r in enumerate(raw):
        doc = next((r[k] for k in ("document", "text", "judgment", "Document", "doc") if k in r and r[k]), "")
        rid = next((r[k] for k in ("id", "doc_id", "ID", "document_id") if k in r), i)
        s = next((r[k] for k in ("summary", "Summary", "reference", "reference_summary") if k in r), None)
        recs.append(dict(id=str(rid), raw_id=rid, document=str(doc), summary=None if s is None else str(s)))
    return recs

TRAIN_JSON, TEST_JSON = p6_find("train.json"), p6_find("test.json")
assert TRAIN_JSON and TEST_JSON, "attach the Phase 6 dataset (Phase_6_Summ/Train/train.json and Test/test.json)"
_all_train = [r for r in p6_read_records(TRAIN_JSON) if r["document"].strip() and r["summary"] and r["summary"].strip()]
test_recs = p6_read_records(TEST_JSON)
random.Random(P6_SEED).shuffle(_all_train)
_dev_n = 16 if P6_DRY_RUN else min(P6_DEV_N, max(20, len(_all_train) // 10))
dev_recs, train_recs = _all_train[:_dev_n], _all_train[_dev_n:]
if P6_DRY_RUN: train_recs, test_recs = train_recs[:160], test_recs[:12]
assert len({r["id"] for r in test_recs}) == len(test_recs), "duplicate test ids"
_dw = [len(r["document"].split()) for r in train_recs]; _sw = [len(r["summary"].split()) for r in train_recs]
print(f"train {len(train_recs)} | dev {len(dev_recs)} | test {len(test_recs)}")
print("document words  p10/p50/p90:", np.percentile(_dw, [10, 50, 90]).astype(int).tolist())
print("summary  words  p10/p50/p90:", np.percentile(_sw, [10, 50, 90]).astype(int).tolist())

In [ ]:
# 4. Text utilities: tokenizer helpers, special markers, long-document selection strategies
def p6_inner_hf():
    objs = [tokenizer] + [getattr(tokenizer, n, None) for n in ("tok", "tokenizer", "_tok", "bpe", "hf")] + list(getattr(tokenizer, "__dict__", {}).values())
    for o in objs:
        if o is not None and hasattr(o, "token_to_id") and hasattr(o, "decode"): return o
    return None

def enc_text(t):
    return [int(i) for i in tokenizer.encode_words(pretokenize(t, PREPROC))]

def p6_special(cands):
    hf = p6_inner_hf()
    if hf is None: return None
    for c in cands:
        i = hf.token_to_id(c)
        if i is not None: return int(i)
    return None

def decode_ids(ids):
    ids = [int(i) for i in ids]
    hf = p6_inner_hf()
    if hf is not None:
        try: return hf.decode(ids)
        except Exception: pass
    for n in ("decode", "decode_ids", "detokenize"):
        f = getattr(tokenizer, n, None)
        if f is not None:
            try: return f(ids)
            except Exception: pass
    inv = getattr(tokenizer, "id_to_token", None)
    if inv is not None: return " ".join(str(inv(i)) for i in ids)
    raise RuntimeError("no decode function found on the Phase-4 tokenizer; see p6_inner_hf / decode_ids")

def p6_detok(t):
    t = re.sub(r"\s+([.,;:!?)\]])", r"\1", t); t = re.sub(r"([(\[])\s+", r"\1", t); t = re.sub(r"\s+", " ", t).strip()
    return re.sub(r"(^|[.!?]\s+)([a-z])", lambda m: m.group(1) + m.group(2).upper(), t)

_SENT_RE = re.compile(r"(?<=[.!?;])\s+(?=[A-Z0-9\"(\[])|\n+")
_STOP = set("a an the of to in and or for on at by with from that this these those is are was were be been as it its which who whom whose he she they his her their them not no but if then than so such any all also may shall would should could can has have had do does did there here into under upon within without".split())
_CUES = re.compile(r"\b(held|dismiss|allow|appeal|convict|acquit|sentenc|conclude|therefore|accordingly|ordered|directed|quash|set aside|upheld|section|article|act)\w*", re.I)

def p6_sentences(text, max_sents=2500):
    out = [s.strip() for s in _SENT_RE.split(text) if s and s.strip()]
    return out[:max_sents]

def _words(s): return [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in _STOP and len(w) > 1]

def _sumbasic_pick(cand, wl, lens, budget, prob, cue):
    """Greedy SumBasic with redundancy update (probabilities of picked words are squared)."""
    prob = dict(prob); chosen = []; used = 0
    cand = list(cand)
    while cand:
        best, bs = None, -1.0
        for i in cand:
            w = wl[i]
            if not w or lens[i] > budget - used: continue
            sc = sum(prob.get(x, 0.0) for x in w) / len(w) + cue[i]
            if sc > bs: best, bs = i, sc
        if best is None: break
        chosen.append(best); used += lens[best]; cand.remove(best)
        for x in set(wl[best]): prob[x] = prob.get(x, 0.0) ** 2
    return chosen

def p6_select(sents, lens, budget, strategy="select"):
    """Indices (original order) of the sentences kept under a token budget."""
    n = len(sents)
    if n == 0: return []
    def take(rng, b):
        out, used = [], 0
        for i in rng:
            if used + lens[i] <= b: out.append(i); used += lens[i]
            elif used >= b * 0.9: break
        return out
    if sum(lens) <= budget: return list(range(n))
    if strategy == "lead": return take(range(n), budget)
    if strategy == "headtail":
        h = take(range(n), int(budget * 0.6)); hs = set(h)
        t = [i for i in take(range(n - 1, -1, -1), budget - sum(lens[i] for i in h)) if i not in hs]
        return sorted(set(h) | set(t))
    wl = [_words(s) for s in sents]; cnt = collections.Counter(w for x in wl for w in x); tot = max(1, sum(cnt.values()))
    prob = {w: c / tot for w, c in cnt.items()}
    cue = [0.002 * min(3, len(_CUES.findall(s))) for s in sents]
    init = sorted(range(n), key=lambda i: -(sum(prob.get(x, 0.0) for x in wl[i]) / max(1, len(wl[i]))))[:250]
    if strategy == "spread":
        S = 6; chosen = []; seg = [(k * n // S, (k + 1) * n // S) for k in range(S)]
        for a, b in seg:
            pool = [i for i in range(a, b)]; pool = sorted(pool, key=lambda i: -(sum(prob.get(x, 0.0) for x in wl[i]) / max(1, len(wl[i]))))[:60]
            chosen += _sumbasic_pick(pool, wl, lens, budget // S, prob, cue)
        return sorted(set(chosen))
    # "select": lead + tail (judgments end with the operative order) + SumBasic over the middle
    lead = take(range(n), int(budget * 0.25)); ls = set(lead)
    tail = [i for i in take(range(n - 1, -1, -1), int(budget * 0.25)) if i not in ls]; ts = set(tail)
    rest = budget - sum(lens[i] for i in lead) - sum(lens[i] for i in tail)
    mid = _sumbasic_pick([i for i in init if i not in ls and i not in ts], wl, lens, rest, prob, cue)
    return sorted(set(lead) | set(tail) | set(mid))

P6_SENT_CAP = 80
def p6_doc_ids(text, strategy, doc_len, pad_id):
    """Selected, order-preserving token ids of the document, left-padded to exactly doc_len."""
    sents = p6_sentences(text)
    ids = [enc_text(s)[:P6_SENT_CAP] for s in sents]
    keep = p6_select(sents, [len(x) for x in ids], doc_len, strategy)
    flat = [t for i in keep for t in ids[i]][:doc_len]
    return [pad_id] * (doc_len - len(flat)) + flat, [sents[i] for i in keep]

def p6_dedup(text):
    seen, out = set(), []
    for s in re.split(r"(?<=[.!?])\s+", text):
        k = re.sub(r"\W+", " ", s.lower()).strip()
        if k and k not in seen: seen.add(k); out.append(s)
    return " ".join(out)

In [ ]:
# 5. Special markers, sequence layout, pre-tokeniser / forward-variant diagnosis on Phase-6 text, prompt cache
#    sequence = [ selected document tokens, left-padded to DOC_LEN ] [SEP] [summary] [END] [pad]
#    Prompts all have the same length, so batches need no attention mask (the Phase-1 models take only token ids).
PAD = int(globals().get("PAD_ID", 0))
_sep = p6_special(["<sep>", "[SEP]", "<|sep|>", "<summary>"])
_end = p6_special(["</s>", "<eos>", "<|endoftext|>", "[EOS]", "<end>"])
SEP_IDS = [_sep] if _sep is not None else [int(i) for i in tokenizer.encode_words(["summary", ":"])]
END_IDS = [_end] if _end is not None else [int(i) for i in tokenizer.encode_words(["endofsummary"])]
print("SEP ids", SEP_IDS, "| END ids", END_IDS, "(special tokens)" if (_sep is not None or _end is not None) else "(word markers)")

# --- pre-tokeniser and forward variants (same search as Phase 4, scored on Phase-6 text) -----------------------
if P4_DIAG and "preproc" in P4_DIAG: PREPROC = P4_DIAG["preproc"]
def p6_ppl_sample(mode, n=24, seed=0):
    rng = random.Random(seed)
    for L in (256, 128, 64):
        rows = []
        for r in rng.sample(train_recs, min(400, len(train_recs))):
            w = pretokenize(r["document"], mode)
            if len(w) < L * 2: continue
            mid = len(w) // 2; ids = tokenizer.encode_words(w[mid:mid + L + L // 2])[:L]
            if len(ids) == L: rows.append(ids)
            if len(rows) >= n: break
        if rows: return torch.tensor(rows, dtype=torch.long)
    raise RuntimeError("documents too short for the perplexity check")

def p6_free():
    gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()

VARIANT_GRID = {
    3: [{"kv": kv, "act": a} for kv in ("interleave", "repeat") for a in ("gelu", "gelu_tanh", "relu")],
    2: [{"rope": r, "qkn": q, "swap": s} for r in ("half", "pairs") for q in ("before", "after") for s in (False, True)],
    1: [{"act": a} for a in ("gelu", "gelu_tanh", "relu")],
}
P6_PPL = {}
if PREPROC == "auto":
    _smp = {m: p6_ppl_sample(m) for m in PREPROC_MODES}; _tab = {}
    for a in (4, 1):
        bb = build_backbone(a, f"Arch{a} (pre-tokeniser check)")
        _tab[a] = {m: lm_ppl(bb, _smp[m]) for m in PREPROC_MODES}; del bb; p6_free()
    PREPROC = min(PREPROC_MODES, key=lambda m: sum(math.log(_tab[a][m]) for a in _tab))
    print("pre-tokeniser table:", {a: {m: round(v, 1) for m, v in d.items()} for a, d in _tab.items()}, "->", PREPROC)
PPL_IDS6 = p6_ppl_sample(PREPROC)
_saved_opts = (P4_DIAG or {}).get("arch_opts") or (P4_DIAG or {}).get("variants") or {}
for a in (4, 3, 2, 1):
    if not globals().get(f"ARCH{a}_CHECKPOINT"): print(f"Arch{a}: checkpoint missing"); continue
    if str(a) in _saved_opts and a in VARIANT_GRID:
        ARCH_OPTS[a] = dict(_saved_opts[str(a)]); print(f"Arch{a}: variant from diagnostics.json", ARCH_OPTS[a]); continue
    bb = build_backbone(a, f"Arch{a} (variant check)")
    if a in VARIANT_GRID:
        res = []
        for opt in VARIANT_GRID[a]:
            ARCH_OPTS[a] = dict(opt); res.append((lm_ppl(bb, PPL_IDS6), opt))
        res.sort(key=lambda r: r[0]); ARCH_OPTS[a] = dict(res[0][1])
    P6_PPL[a] = lm_ppl(bb, PPL_IDS6); print(f"Arch{a}: Phase-6 text perplexity {P6_PPL[a]:.1f} {ARCH_OPTS.get(a, '')}")
    del bb; p6_free()
print("PREPROC =", PREPROC)

# --- sequence layout -----------------------------------------------------------------------------------------
_stok = [len(enc_text(r["summary"])) for r in random.Random(1).sample(train_recs, min(400, len(train_recs)))]
SUM_LEN = int(np.clip(np.percentile(_stok, 75), 48, 192))
CTX = min(512, int(globals().get("MAX_POS", 512)))
DOC_LEN = CTX - SUM_LEN - len(SEP_IDS) - len(END_IDS)
P_LEN = DOC_LEN + len(SEP_IDS)               # prompt length
T_SUM = CTX - P_LEN                          # tokens available for summary + END
print(f"summary tokens p50/p75/p90: {np.percentile(_stok,[50,75,90]).astype(int).tolist()} -> SUM_LEN {SUM_LEN} | CTX {CTX} | document budget {DOC_LEN} tokens")
_t = train_recs[0]["summary"]; _rt = decode_ids(enc_text(_t)[:40])
print("decode round-trip check:\n   ref :", _t[:150].replace("\n", " "), "\n   back:", p6_detok(_rt)[:150])

# --- prompt cache ----------------------------------------------------------------------------------------------
P6_CACHE = {}
def p6_prompts(recs, strategy, tag):
    key = (tag, strategy, DOC_LEN, PREPROC, len(recs))
    if key in P6_CACHE: return P6_CACHE[key]
    f = f"{P6_OUT}/prompts_{tag}_{strategy}_{DOC_LEN}_{len(recs)}_{'dry' if P6_DRY_RUN else 'full'}.pkl"
    if os.path.exists(f):
        P6_CACHE[key] = pickle.load(open(f, "rb")); return P6_CACHE[key]
    t0 = time.time(); arr = np.zeros((len(recs), P_LEN), dtype=np.int32); sel = []
    for i, r in enumerate(recs):
        ids, ss = p6_doc_ids(r["document"], strategy, DOC_LEN, PAD)
        arr[i, :DOC_LEN] = ids; arr[i, DOC_LEN:] = SEP_IDS; sel.append(ss)
        if i and i % 500 == 0: print(f"   prompts {tag}/{strategy}: {i}/{len(recs)}  {time.time()-t0:.0f}s")
    P6_CACHE[key] = (arr, sel); pickle.dump(P6_CACHE[key], open(f, "wb"))
    print(f"prompts {tag}/{strategy}: {len(recs)} docs in {time.time()-t0:.0f}s"); return P6_CACHE[key]

def p6_train_tensors(strategy):
    arr, _ = p6_prompts(train_recs, strategy, "train")
    X = np.full((len(train_recs), CTX), PAD, dtype=np.int64); sl = np.zeros(len(train_recs), dtype=np.int64)
    X[:, :P_LEN] = arr
    for i, r in enumerate(train_recs):
        s = (enc_text(r["summary"])[:SUM_LEN] + END_IDS)[:T_SUM]; X[i, P_LEN:P_LEN + len(s)] = s; sl[i] = len(s)
    return torch.from_numpy(X), torch.from_numpy(sl)

def p6_dev_tensors(strategy):
    arr, _ = p6_prompts(dev_recs, strategy, "dev")
    X = np.full((len(dev_recs), CTX), PAD, dtype=np.int64); sl = np.zeros(len(dev_recs), dtype=np.int64)
    X[:, :P_LEN] = arr
    for i, r in enumerate(dev_recs):
        s = (enc_text(r["summary"])[:SUM_LEN] + END_IDS)[:T_SUM]; X[i, P_LEN:P_LEN + len(s)] = s; sl[i] = len(s)
    return torch.from_numpy(X), torch.from_numpy(sl)

In [ ]:
# 6. Adaptation techniques (all start from the Phase-1 weights) and the LM wrapper
#    LoRMA   : multiplicative low-rank update  W' = (I + B A) W   (B = 0 at start -> identical to Phase 1)
#    LoRA    : additive low-rank update        W' = W + B A
#    Houlsby : bottleneck adapter after the attention and after the FFN sub-layer of every block
#    KD      : full fine-tuning of a half-depth student against a stronger teacher's distribution
class P6LoRALinear(nn.Module):
    def __init__(self, base, r=16, alpha=32, mode="lora", dropout=0.0):
        super().__init__()
        self.base, self.mode, self.scale = base, mode, alpha / r
        d_in = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, d_in)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r)); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = self.base(x); z = x if self.mode == "lora" else y     # lorma: low-rank map of the layer OUTPUT (multiplicative)
        return y + self.scale * F.linear(F.linear(self.drop(z), self.A), self.B)

class P6Bottleneck(nn.Module):
    def __init__(self, d, r=64):
        super().__init__(); self.down, self.up = nn.Linear(d, r), nn.Linear(r, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return x + self.up(F.gelu(self.down(x)))

class P6AdaptedSublayer(nn.Module):
    def __init__(self, sub, d, r):
        super().__init__(); self.sub, self.ad = sub, P6Bottleneck(d, r)
    def forward(self, *a, **k):
        o = self.sub(*a, **k)
        return (self.ad(o[0]),) + tuple(o[1:]) if isinstance(o, tuple) else self.ad(o)

def p6_replace_linears(root, make):
    for name, m in [(n, m) for n, m in root.named_modules() if isinstance(m, nn.Linear)]:
        parent = root; parts = name.split(".")
        for p in parts[:-1]: parent = getattr(parent, p)
        setattr(parent, parts[-1], make(m))

def p6_add_houlsby(bb, r=64):
    ATT, FFN = {"attn", "attention", "self_attn", "self_attention"}, {"ffn", "mlp", "feed_forward", "ff"}
    n = 0
    for blk in bb.blocks:
        for cname, child in list(blk.named_children()):
            if cname.lower() in ATT | FFN:
                d = next(m for m in child.modules() if isinstance(m, nn.Linear)).in_features
                setattr(blk, cname, P6AdaptedSublayer(child, d, r)); n += 1
    assert n, f"no attention/FFN sub-layers found; block children: {[c for c, _ in bb.blocks[0].named_children()]}"
    return n

class P6LM(nn.Module):
    def __init__(self, bb): super().__init__(); self.bb = bb
    def hidden(self, ids):
        h = self.bb(ids); return h[0] if isinstance(h, (tuple, list)) else h
    def _head(self, h):
        with torch.autocast(device_type=device.type, enabled=False): return F.linear(h.float(), lm_weight(self.bb))
    def logits_slice(self, ids, P): return self._head(self.hidden(ids)[:, P - 1:ids.size(1) - 1])
    def last_logits(self, ids): return self._head(self.hidden(ids)[:, -1])

def p6_make_model(arch, tech, cfg, name):
    bb = build_backbone(arch, name)
    if cfg.get("keep"):                                   # half-depth student
        n_keep = max(1, int(round(len(bb.blocks) * cfg["keep"]))); bb.blocks = nn.ModuleList(list(bb.blocks)[:n_keep])
        print(f"  {name}: kept the first {n_keep} blocks")
    if tech in ("kd", "full"):
        for p in bb.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if tech in ("lora", "lorma"):
            p6_replace_linears(bb.blocks, lambda m: P6LoRALinear(m, cfg.get("r", 16), 2 * cfg.get("r", 16), tech, 0.05))
        elif tech == "houlsby": print("  houlsby sub-layers wrapped:", p6_add_houlsby(bb, cfg.get("bott", 64)))
    return P6LM(bb).to(device)

def p6_count(model):
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr

In [ ]:
# 7. Training engine (time-budgeted, OOM-safe), distillation loss, generation, metrics
def p6_loss(model, ids, sl, teacher=None, alpha=0.5, T=2.0, amp=True):
    with torch.autocast(device_type=device.type, dtype=P6_AMP_DTYPE, enabled=amp):
        lg = model.logits_slice(ids, P_LEN)
    tgt = ids[:, P_LEN:]; m = (torch.arange(tgt.size(1), device=ids.device)[None, :] < sl[:, None]).float(); den = m.sum().clamp(min=1)
    ce = (F.cross_entropy(lg.reshape(-1, lg.size(-1)), tgt.reshape(-1), reduction="none").view_as(m) * m).sum() / den
    if teacher is None: return ce
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=P6_AMP_DTYPE, enabled=amp):
        tl = teacher.logits_slice(ids, P_LEN)
    kd = (F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tl / T, -1), reduction="none").sum(-1) * m).sum() / den * T * T
    return alpha * ce + (1 - alpha) * kd

@torch.no_grad()
def p6_val_nll(model, X, sl, amp=True, bs=16):
    model.eval(); tot = n = 0.0
    for s in range(0, X.size(0), bs):
        ids = X[s:s + bs].to(device); l = sl[s:s + bs].to(device)
        with torch.autocast(device_type=device.type, dtype=P6_AMP_DTYPE, enabled=amp):
            lg = model.logits_slice(ids, P_LEN)
        tgt = ids[:, P_LEN:]; m = (torch.arange(tgt.size(1), device=device)[None, :] < l[:, None]).float()
        ce = F.cross_entropy(lg.reshape(-1, lg.size(-1)), tgt.reshape(-1), reduction="none").view_as(m)
        tot += float((ce * m).sum()); n += float(m.sum())
    return tot / max(1.0, n)

def p6_amp_ok(model, X, sl):
    """Mixed precision is kept only if it reproduces the fp32 loss (Phase 4 found it unsafe for some backbones)."""
    if not P6_AMP_ON: return False
    a, b = p6_val_nll(model, X[:16], sl[:16], amp=False), p6_val_nll(model, X[:16], sl[:16], amp=True)
    ok = math.isfinite(b) and abs(a - b) / max(a, 1e-6) < 0.03
    print(f"  amp check: fp32 {a:.4f} vs amp {b:.4f} -> {'amp on' if ok else 'fp32 (amp unsafe)'}"); return ok

def p6_train(model, X, sl, lr, minutes, wd=0.0, teacher=None, alpha=0.5, T=2.0, max_epochs=8, amp=True, log=25):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd, betas=(0.9, 0.98))
    try: scaler = torch.amp.GradScaler("cuda", enabled=amp and P6_AMP_DTYPE == torch.float16)
    except Exception: scaler = torch.cuda.amp.GradScaler(enabled=amp and P6_AMP_DTYPE == torch.float16)
    N = X.size(0); mb, accum = P6_MICRO_BS, P6_ACCUM
    max_steps = max(1, N // (mb * accum)) * max_epochs; budget = minutes * 60; t0 = time.time()
    step, ema, ptr, epoch = 0, None, 0, 0; perm = torch.randperm(N); model.train(); hist = []
    while step < max_steps and time.time() - t0 < budget:
        prog = max(step / max_steps, (time.time() - t0) / budget)
        lr_now = lr * min(1.0, (step + 1) / 30) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))
        for g in opt.param_groups: g["lr"] = lr_now
        eff = mb * accum
        if ptr + eff > N: perm, ptr, epoch = torch.randperm(N), 0, epoch + 1
        idx = perm[ptr:ptr + eff]
        try:
            opt.zero_grad(set_to_none=True); tot = 0.0
            for j in range(0, len(idx), mb):
                b = idx[j:j + mb]
                loss = p6_loss(model, X[b].to(device), sl[b].to(device), teacher, alpha, T, amp) * (len(b) / len(idx))
                scaler.scale(loss).backward(); tot += float(loss)
            if not math.isfinite(tot): opt.zero_grad(set_to_none=True); ptr += len(idx); continue
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
        except torch.cuda.OutOfMemoryError:
            opt.zero_grad(set_to_none=True); p6_free()
            if mb == 1: raise
            mb, accum = max(1, mb // 2), accum * 2; print(f"   OOM -> micro-batch {mb}, accumulation {accum}"); continue
        ptr += len(idx); step += 1; ema = tot if ema is None else 0.95 * ema + 0.05 * tot
        if step % log == 0 or step == 1:
            hist.append((step, round(ema, 4))); print(f"   step {step:5d} | epoch {epoch} | loss {ema:.4f} | lr {lr_now:.2e} | {(time.time()-t0)/60:.1f} min")
    model.eval(); return dict(steps=step, epochs=round(step * mb * accum / max(1, N), 2), minutes=round((time.time() - t0) / 60, 1), loss_hist=hist)

# --- generation -----------------------------------------------------------------------------------------------
@torch.no_grad()
def p6_generate(model, prompts, max_new=None, min_new=0, bs=None, nrep=None, amp=True):
    """Greedy decoding with no-repeat n-gram blocking. All prompts share one length, so no padding mask is needed."""
    max_new = max_new or T_SUM; bs = bs or P6_GEN_BS; nrep = P6_NO_REPEAT if nrep is None else nrep
    model.eval(); outs = []; k = len(END_IDS)
    for s in range(0, len(prompts), bs):
        ids = torch.as_tensor(np.asarray(prompts[s:s + bs]), dtype=torch.long, device=device); B = ids.size(0)
        gen = [[] for _ in range(B)]; done = [False] * B; ban = [dict() for _ in range(B)]
        for t in range(max_new):
            with torch.autocast(device_type=device.type, dtype=P6_AMP_DTYPE, enabled=amp):
                lg = model.last_logits(ids)
            if k == 1 and t < min_new: lg[:, END_IDS[0]] = -1e9
            if nrep > 1:
                for b in range(B):
                    if done[b] or len(gen[b]) < nrep - 1: continue
                    bad = ban[b].get(tuple(gen[b][len(gen[b]) - (nrep - 1):]))
                    if bad: lg[b, list(bad)] = -1e9
            nxt = lg.argmax(-1); ids = torch.cat([ids, nxt[:, None]], 1); nl = nxt.tolist()
            for b in range(B):
                if done[b]: continue
                gen[b].append(nl[b])
                if nrep > 1 and len(gen[b]) >= nrep: ban[b].setdefault(tuple(gen[b][-nrep:-1]), set()).add(gen[b][-1])
                if gen[b][-k:] == END_IDS: done[b] = True
            if all(done): break
        outs += gen
    return outs

def p6_post(tok_ids, fallback=""):
    tok_ids = list(tok_ids); k = len(END_IDS)
    for i in range(len(tok_ids) - k + 1):
        if tok_ids[i:i + k] == END_IDS: tok_ids = tok_ids[:i]; break
    txt = p6_dedup(p6_detok(decode_ids(tok_ids))) if tok_ids else ""
    return txt if txt.strip() else fallback

# --- metrics ----------------------------------------------------------------------------------------------------
def _tk(s): return re.findall(r"[a-z0-9]+", s.lower())
def _f1(p, r, h): return 0.0 if h == 0 or p == 0 else 2 * p * r / (p + r)
def rouge1_f(ref, hyp):
    a, b = collections.Counter(_tk(ref)), collections.Counter(_tk(hyp)); ov = sum((a & b).values())
    return _f1(ov / max(1, sum(b.values())), ov / max(1, sum(a.values())), ov)
def rougeL_f(ref, hyp):
    a, b = _tk(ref), _tk(hyp)
    if not a or not b: return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b): cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]; return _f1(l / len(b), l / len(a), l)
def rep_ngram(t, n=3):
    w = _tk(t); g = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 0.0 if not g else 1 - len(set(g)) / len(g)
def num_recall(ref, hyp):
    a = set(re.findall(r"\d+", ref)); return 1.0 if not a else len(a & set(re.findall(r"\d+", hyp))) / len(a)

P6_BERT = {"ok": None}
def bertscore_f1(hyps, refs):
    if P6_BERT["ok"] is False: return None
    try:
        from bert_score import score
        _, _, f = score(hyps, refs, lang="en", model_type=P6_BERT_MODEL, verbose=False, device=str(device)); P6_BERT["ok"] = True
        return float(f.mean())
    except Exception as e:
        if P6_BERT["ok"] is None: print("  BERTScore unavailable here (needs internet + bert-score):", repr(e)[:120], "-> ROUGE only")
        P6_BERT["ok"] = False; return None

def p6_score(refs, hyps):
    d = dict(rouge1=float(np.mean([rouge1_f(r, h) for r, h in zip(refs, hyps)])), rougeL=float(np.mean([rougeL_f(r, h) for r, h in zip(refs, hyps)])),
             rep3=float(np.mean([rep_ngram(h) for h in hyps])), num_recall=float(np.mean([num_recall(r, h) for r, h in zip(refs, hyps)])),
             words=float(np.mean([len(h.split()) for h in hyps])))
    d["bertscore"] = bertscore_f1(hyps, refs); return d

def p6_extractive_baseline():
    _, sel = p6_prompts(dev_recs, P6_STRATEGY, "dev")
    hyps = [" ".join(s)[: SUM_LEN * 6] for s in sel]; return p6_score([r["summary"] for r in dev_recs], hyps)

In [ ]:
# 8. Runs. One run per Phase-1 architecture, each with a different technique.
#    Run 1  Arch 4  LoRMA (compulsory)           also the KD teacher
#    Run 2  Arch 3  Houlsby bottleneck adapters
#    Run 3  Arch 2  LoRA
#    Run 4  Arch 1  Knowledge distillation       half-depth student, full fine-tuning, loss = CE + KL to the Run-1 teacher
#    Run 5  Arch 1  half-depth student, full fine-tuning WITHOUT teacher (optional; isolates what distillation adds)
P6_RUN_CFG = {
    1: dict(name="Run1_Arch4_LoRMA",     arch=4, tech="lorma",   r=16,    lr=6e-4),
    2: dict(name="Run2_Arch3_Houlsby",   arch=3, tech="houlsby", bott=64, lr=1e-3),
    3: dict(name="Run3_Arch2_LoRA",      arch=2, tech="lora",    r=16,    lr=6e-4),
    4: dict(name="Run4_Arch1_KD",        arch=1, tech="kd",      keep=0.5, lr=3e-4, wd=0.01, alpha=0.5, T=2.0),
    5: dict(name="Run5_Arch1_FullFT_noKD", arch=1, tech="full",  keep=0.5, lr=3e-4, wd=0.01),
}

def p6_teacher():
    t = p6_make_model(4, "lorma", dict(r=16), "Teacher: Arch 4 + LoRMA (Run 1)")
    f = next(iter(glob.glob(f"{P6_OUT}/Run1_*_trainable.pt") + glob.glob("/kaggle/input/**/Run1_*_trainable.pt", recursive=True)), None)
    assert f, "teacher weights not found: run Run 1 first, or attach the committed output containing Run1_Arch4_LoRMA_trainable.pt"
    res = t.load_state_dict(torch.load(f, map_location="cpu"), strict=False)
    assert not [k for k in res.missing_keys if k.endswith((".A", ".B"))], "teacher adapter weights did not load"
    for p in t.parameters(): p.requires_grad = False
    print("  teacher loaded from", f); return t.eval()

def p6_gen_eval(model, recs, strategy, amp, tag, with_lengths=False):
    prompts, sel = p6_prompts(recs, strategy, tag)
    t0 = time.time(); gens = p6_generate(model, prompts, min_new=int(0.3 * SUM_LEN) if len(END_IDS) == 1 else 0, amp=amp); sec = time.time() - t0
    fb = [" ".join(s[:2]) for s in sel]
    hyps = [p6_post(g, fb[i]) for i, g in enumerate(gens)]
    out = dict(hyps=hyps, gens=gens, sec_per_doc=sec / max(1, len(recs)))
    if with_lengths:
        refs = [r["summary"] for r in recs]; out["by_length"] = {}
        for L in (32, 64, 96, 128, T_SUM):
            if L <= T_SUM: out["by_length"][L] = p6_score(refs, [p6_post(g[:L], fb[i]) for i, g in enumerate(gens)])
    return out

def p6_run(rid):
    cfg = P6_RUN_CFG[rid]; name = cfg["name"]; sub = f"{P6_OUT}/submission_{name}.zip"; mf = f"{P6_OUT}/{name}_metrics.json"
    if P6_SKIP_DONE and os.path.exists(sub) and os.path.exists(mf):
        print(f"[{name}] already done, skipping"); return json.load(open(mf))
    print("\n" + "=" * 100 + f"\n{name}: Arch {cfg['arch']} + {cfg['tech']}\n" + "=" * 100)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    model = p6_make_model(cfg["arch"], cfg["tech"], cfg, name); tot, trn = p6_count(model)
    print(f"  parameters: {tot/1e6:.1f}M total | {trn/1e6:.2f}M trainable ({100*trn/tot:.1f}%)")
    X, sl = p6_train_tensors(P6_STRATEGY); Xv, slv = p6_dev_tensors(P6_STRATEGY)
    amp = p6_amp_ok(model, Xv, slv); nll0 = p6_val_nll(model, Xv, slv, amp); print(f"  Dev NLL before adaptation: {nll0:.4f}")
    teacher = p6_teacher() if cfg["tech"] == "kd" else None
    info = p6_train(model, X, sl, cfg["lr"], 3 if P6_DRY_RUN else P6_MINUTES[rid], cfg.get("wd", 0.0), teacher, cfg.get("alpha", 0.5), cfg.get("T", 2.0), amp=amp)
    del teacher; p6_free()
    nll1 = p6_val_nll(model, Xv, slv, amp); print(f"  Dev NLL after adaptation : {nll1:.4f}")
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    refs = [r["summary"] for r in dev_recs]
    dv = p6_gen_eval(model, dev_recs, P6_STRATEGY, amp, "dev", with_lengths=True); dsc = p6_score(refs, dv["hyps"])
    print("  DEV", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in dsc.items()}); print("  example:", dv["hyps"][0][:300])
    strat = {}
    if rid == 1 and P6_STRATEGY_ABLATION and not P6_DRY_RUN:
        for st in [s for s in ("select", "spread", "headtail", "lead") if s != P6_STRATEGY]:
            g = p6_gen_eval(model, dev_recs, st, amp, "dev"); strat[st] = p6_score(refs, g["hyps"]); print(f"  strategy {st}:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in strat[st].items()})
    ts = p6_gen_eval(model, test_recs, P6_STRATEGY, amp, "test")
    preds = [dict(id=r["raw_id"], summary=ts["hyps"][i]) for i, r in enumerate(test_recs)]
    assert len(preds) == len(test_recs) and all(p["summary"].strip() for p in preds), "empty or missing summary"
    json.dump(preds, open(f"{P6_OUT}/{name}_predictions.json", "w"), ensure_ascii=False)
    with zipfile.ZipFile(sub, "w", zipfile.ZIP_DEFLATED) as z: z.write(f"{P6_OUT}/{name}_predictions.json", "predictions.json")
    json.dump([dict(id=r["id"], reference=r["summary"], summary=dv["hyps"][i]) for i, r in enumerate(dev_recs)], open(f"{P6_OUT}/{name}_dev_predictions.json", "w"), ensure_ascii=False)
    if cfg["tech"] not in ("kd", "full"):
        torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, f"{P6_OUT}/{name}_trainable.pt")
    m = dict(run=name, arch=cfg["arch"], technique=cfg["tech"], strategy=P6_STRATEGY, params_total_M=round(tot / 1e6, 2), params_trainable_M=round(trn / 1e6, 3),
             peak_gpu_gb=round(peak, 2), amp=amp, dev_nll_before=round(nll0, 4), dev_nll_after=round(nll1, 4), train=info, dev=dsc, dev_by_length=dv["by_length"],
             strategy_ablation=strat, test_sec_per_doc=round(ts["sec_per_doc"], 3), dev_sec_per_doc=round(dv["sec_per_doc"], 3), ctx=CTX, doc_tokens=DOC_LEN, sum_tokens=SUM_LEN, dry=P6_DRY_RUN)
    json.dump(m, open(mf, "w"), indent=1, default=str); print(f"  wrote {sub}")
    del model, X, sl, Xv, slv; p6_free(); return m

P6_RESULTS = {}
for _rid in sorted(P6_RUNS):
    P6_RESULTS[_rid] = p6_run(_rid)

In [ ]:
# 9. Comparison table, analysis notes and submission check (reads every *_metrics.json in the output folder,
import pandas as pd
_ms = {}
for f in glob.glob(f"{P6_OUT}/*_metrics.json") + glob.glob("/kaggle/input/**/p6_out/*_metrics.json", recursive=True):
    m = json.load(open(f)); _ms.setdefault(m["run"], m)
rows = []
for n, m in sorted(_ms.items()):
    d = m["dev"]; rows.append({"run": n, "arch": m["arch"], "technique": m["technique"], "params_M": m["params_total_M"], "trainable_M": m["params_trainable_M"],
        "peak_GB": m["peak_gpu_gb"], "train_min": m["train"]["minutes"], "epochs": m["train"]["epochs"], "NLL_before": m["dev_nll_before"], "NLL_after": m["dev_nll_after"],
        "BERTScore": d.get("bertscore"), "R1": round(d["rouge1"], 4), "RL": round(d["rougeL"], 4), "rep3": round(d["rep3"], 3), "num_recall": round(d["num_recall"], 3),
        "words": round(d["words"], 1), "s_per_doc": m["test_sec_per_doc"]})
df = pd.DataFrame(rows); pd.set_option("display.width", 250, "display.max_columns", 30); print(df.to_string(index=False))
if not P6_DRY_RUN:
    try: print("\nExtractive reference (selected sentences, no model):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in p6_extractive_baseline().items()})
    except Exception as e: print("extractive baseline skipped:", repr(e)[:100])
lines = ["# Phase 6 analysis notes (numbers from the Dev split cut from train.json)\n", df.to_markdown(index=False) if hasattr(df, "to_markdown") else df.to_string(index=False), ""]
if len(df):
    key = "BERTScore" if df["BERTScore"].notna().any() else "RL"; best = df.sort_values(key, ascending=False).iloc[0]
    lines.append(f"- Best run on Dev by {key}: {best['run']} ({best[key]}).")
    for n, m in sorted(_ms.items()):
        bl = m.get("dev_by_length", {})
        if bl: lines.append(f"- {n}: ROUGE-L by generation cap " + ", ".join(f"{L} tok -> {v['rougeL']:.3f}" for L, v in bl.items()) + ".")
        if m.get("strategy_ablation"): lines.append(f"- {n}: long-document strategies at inference (trained on '{m['strategy']}'): " + ", ".join(f"{s} RL {v['rougeL']:.3f}" for s, v in m["strategy_ablation"].items()) + f", {m['strategy']} RL {m['dev']['rougeL']:.3f}.")
    r4, r5, r1 = (_ms.get(n) for n in ("Run4_Arch1_KD", "Run5_Arch1_FullFT_noKD", "Run1_Arch4_LoRMA"))
    if r4 and r1: lines.append(f"- KD: student {r4['params_total_M']}M params vs teacher {r1['params_total_M']}M; {r4['test_sec_per_doc']}s/doc vs {r1['test_sec_per_doc']}s/doc; Dev RL {r4['dev']['rougeL']:.3f} vs {r1['dev']['rougeL']:.3f}.")
    if r4 and r5: lines.append(f"- Distillation vs the same student without a teacher: RL {r4['dev']['rougeL']:.3f} vs {r5['dev']['rougeL']:.3f}.")
open(f"{P6_OUT}/analysis_summary.md", "w").write("\n".join(lines)); print("\n".join(lines[2:]))

_ids = {str(r["id"]) for r in test_recs}
for z in sorted(glob.glob(f"{P6_OUT}/submission_*.zip")):
    p = json.load(zipfile.ZipFile(z).open("predictions.json"))
    _pid = {str(x["id"]) for x in p}
    print(f"{os.path.basename(z)}: {len(p)} summaries | ids match test: {_pid == _ids} | empty: {sum(not x['summary'].strip() for x in p)}")
print("\nSubmit the four zips Run1..Run4 (list format, one summary per test id). Run5 is for the analysis only.")